# Incident extraction with LLaMA through Groq
Run cells in order. Source code stays in Python modules. Demo documents are synthetic.
API calls send incident text to Groq; use only data you are authorized to send.


In [ ]:
from google.colab import drive
from pathlib import Path
import sys

drive.mount('/content/drive')
# Change this relative folder to where you placed the project in My Drive.
project_relative_path = 'incident_extraction'
project_dir = Path('/content/drive/MyDrive') / project_relative_path
if not (project_dir / 'requirements.txt').is_file():
    raise FileNotFoundError('Set project_relative_path to your incident_extraction folder')
# Import the folder as a package, including when its parent is on Google Drive.
sys.path.insert(0, str(project_dir.parent))


In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r',
                       str(project_dir / 'requirements.txt')])


## Secure credentials
In Colab's Secrets panel, create `GROQ_API_KEY` and enable notebook access.
Never paste the key into a cell or save it in Drive. The Python module itself reads
`GROQ_API_KEY` from the environment and has no Colab dependency.


In [ ]:
import os
from google.colab import userdata

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
if not os.environ.get('GROQ_API_KEY'):
    raise RuntimeError('Provide GROQ_API_KEY in Colab Secrets')


In [ ]:
from incident_extraction.extraction import process_documents
from incident_extraction.schemas import IncidentData
import json

# Replace demo_claims.json with team records using the documented format.
# Existing results are protected. Choose another subfolder for a fresh run.
output_dir = project_dir / 'outputs'
saved = process_documents(project_dir / 'data' / 'demo_claims.json', output_dir)
for path in saved:
    result = json.loads(path.read_text(encoding='utf-8'))
    IncidentData.model_validate(result['incident'])
    print(path.name)
    print(json.dumps(result, indent=2, ensure_ascii=False))


## Offline checks (no Groq requests)
These tests validate schemas, retries and output saving with mocked responses.
They do not validate actual LLaMA extraction quality.


In [ ]:
subprocess.check_call([sys.executable, '-m', 'unittest', 'discover',
                       '-s', str(project_dir / 'tests'), '-v'], cwd=project_dir.parent)
